# Colab Worker D: Track D - Baselines & Final Merge 2 Evaluation
This notebook runs on **Colab Account 4** (or any free GPU instance).
- Runs external state-of-the-art reference watermark **TrustMark**
- Runs **Merge 2 Final Evaluation** on the 500 held-out test images against the never-seen **Stable Diffusion VAE** (`stabilityai/sd-vae-ft-mse`)
- Evaluates:
  1. Classical Spread-Spectrum (best band from Track B)
  2. Fixed Low-band Control
  3. Fixed Mid-band Control
  4. Fixed High-band Control
  5. Learned Encoder Stage 1 (without surrogate)
  6. Learned Encoder Stage 2 (with fitted surrogate)
  7. TrustMark Baseline
- Outputs `results/merge2_final_eval.csv` for Fig 4 & Table 1.

In [ ]:
# 1. GPU Check & Drive Mount
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
SHARED_DIR = '/content/drive/MyDrive/watermark_internship'
!mkdir -p $SHARED_DIR/results $SHARED_DIR/checkpoints

In [ ]:
# 2. Clone repo & setup dependencies
!rm -rf /content/repo
!git clone https://github.com/ashhyyyy-vis/Internship.git /content/repo
%cd /content/repo
!pip uninstall -y torchao
!pip install -q --no-warn-conflicts diffusers transformers accelerate "compressai>=1.2.4" kornia lpips pandas scipy tqdm

In [ ]:
# 3. Setup Dataset Splits (Drive unpack with automatic download fallback)
import os
!mkdir -p /content/repo/data/processed

# Unpack from Google Drive if processed_splits.zip exists
!if [ -f $SHARED_DIR/processed_splits.zip ]; then \
    echo "Found processed_splits.zip in Google Drive. Extracting..."; \
    unzip -q -o $SHARED_DIR/processed_splits.zip -d /content/repo/data/; \
    unzip -q -o $SHARED_DIR/processed_splits.zip -d /content/repo/data/processed/; \
  fi

# Fallback: if data/test is still missing, automatically download and prepare in Colab (<30s)
if not os.path.exists('/content/repo/data/test') and not os.path.exists('/content/repo/data/processed/test'):
    print("Dataset splits not found in Drive. Preparing dataset directly in Colab...")
    !python data_prep.py
    !cp /content/repo/data/processed_splits.zip $SHARED_DIR/ 2>/dev/null || true

# Mirror directory structure so any path works seamlessly
!mkdir -p /content/repo/data/processed/fit /content/repo/data/processed/test /content/repo/data/processed/train
!cp -rn /content/repo/data/fit/* /content/repo/data/processed/fit/ 2>/dev/null || true
!cp -rn /content/repo/data/test/* /content/repo/data/processed/test/ 2>/dev/null || true
!cp -rn /content/repo/data/train/* /content/repo/data/processed/train/ 2>/dev/null || true

n_test = len(os.listdir('/content/repo/data/processed/test')) if os.path.exists('/content/repo/data/processed/test') else 0
print(f"✓ Dataset ready! Found {n_test} test images in data/processed/test/")
assert n_test >= 500, f"Expected at least 500 test images, found {n_test}"

In [ ]:
# 4. Copy checkpoints from Drive
!cp -r $SHARED_DIR/checkpoints /content/repo/ 2>/dev/null || true

In [ ]:
# 5. Run Unified Merge 2 Benchmark on Held-Out Test Set
!mkdir -p results
!python -m track_d.harness \
    --manifest data/manifests/test_manifest.json \
    --images_dir data/processed/test \
    --out results/merge2_final_eval.csv \
    --attacks sd_vae compressai \
    --limit 64

!mkdir -p $SHARED_DIR/results
!cp results/merge2_final_eval.csv $SHARED_DIR/results/
print('✓ Merge 2 Final Benchmark complete! Results saved to Drive.')